Dataset exploration exercise 3

1. Libary and device setup

In [12]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from collections import Counter

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

In [13]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


2. Class count and Disease name

In [14]:
# Load disease label map
with open("/kaggle/input/competitions/cassava-leaf-disease-classification/label_num_to_disease_map.json", "r") as f:
    label_num_to_disease_map = json.load(f)

# Load training data
train_df = pd.read_csv("/kaggle/input/competitions/cassava-leaf-disease-classification/train.csv")

# Count labels
train_counts = Counter(train_df["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    disease_name = label_num_to_disease_map[str(class_idx)]
    print(f"{disease_name}: {count}")


Training set class counts:

Cassava Bacterial Blight (CBB): 1087
Cassava Brown Streak Disease (CBSD): 2189
Cassava Green Mottle (CGM): 2386
Cassava Mosaic Disease (CMD): 13158
Healthy: 2577


3. Train and Split Validation and Distribution

In [15]:
# Split dataset (80% train, 20% validation)
train_df_split, valid_df_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42
)

# Count labels
train_counts = Counter(train_df_split["label"])
valid_counts = Counter(valid_df_split["label"])

print("Training set class counts:\n")
for class_idx, count in sorted(train_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")

print("\nValidation set class counts:\n")
for class_idx, count in sorted(valid_counts.items()):
    print(f"{label_num_to_disease_map[str(class_idx)]}: {count}")

Training set class counts:

Cassava Bacterial Blight (CBB): 870
Cassava Brown Streak Disease (CBSD): 1751
Cassava Green Mottle (CGM): 1909
Cassava Mosaic Disease (CMD): 10526
Healthy: 2061

Validation set class counts:

Cassava Bacterial Blight (CBB): 217
Cassava Brown Streak Disease (CBSD): 438
Cassava Green Mottle (CGM): 477
Cassava Mosaic Disease (CMD): 2632
Healthy: 516


4. SimpleCNN (5 Classes)

In [16]:
from torchvision import models

def build_model(num_classes=5):
    try:
        weights = models.ResNet18_Weights.DEFAULT
    except AttributeError:
        weights = 'DEFAULT'
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [17]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean', ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
        self.ignore_index = ignore_index
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction='none', ignore_index=ignore_index)

    def forward(self, logits, target):
        logpt = -self.ce(logits, target)
        pt = logpt.exp()
        focal_term = (1 - pt) ** self.gamma
        loss = -focal_term * logpt

        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss

model = build_model().to(device)
criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


5. Creating Dataset Class

In [18]:
class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

6. Image Transform

In [19]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

7. Creating Train and Valid dataset

In [20]:
train_dataset = CassavaDataset(
    df=train_df_split,
    image_dir="/kaggle/input/competitions/cassava-leaf-disease-classification/train_images",
    transform=transform
)

valid_dataset = CassavaDataset(
    df=valid_df_split,
    image_dir="/kaggle/input/competitions/cassava-leaf-disease-classification/train_images",
    transform=transform
)


In [21]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

8. Training CNN model

In [22]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total

    model.eval()
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            valid_total += labels.size(0)
            valid_correct += (predicted == labels).sum().item()

    valid_accuracy = 100 * valid_correct / valid_total
    avg_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {avg_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Accuracy: {valid_accuracy:.2f}%\n")

Epoch [1/5]
Training Loss: 0.4615
Training Accuracy: 70.04%
Validation Accuracy: 66.89%

Epoch [2/5]
Training Loss: 0.3295
Training Accuracy: 76.82%
Validation Accuracy: 73.04%

Epoch [3/5]
Training Loss: 0.2854
Training Accuracy: 79.41%
Validation Accuracy: 76.87%

Epoch [4/5]
Training Loss: 0.2485
Training Accuracy: 81.54%
Validation Accuracy: 74.60%

Epoch [5/5]
Training Loss: 0.2158
Training Accuracy: 83.24%
Validation Accuracy: 78.88%

